# Paper 2 — Devanagari Handwriting LDM, Trained From Scratch (One-Shot, Kaggle)

**One run = the full Paper 2 v2 experiment:** train TWO compact latent diffusion
models from scratch (codepoint-conditioned vs **akshara**-conditioned — the paper's
central ablation) on 70k Hindi word images, then sample the conjunct-stratified
evaluation sets (in-vocab / OOV / unseen-conjunct) and score them with FID +
recognizer-judge content fidelity.

Architecture: frozen SD-VAE (64×256 → 4×8×32 latents, precomputed once ≈150MB),
~65M-param UNet with cross-attention over learned token embeddings, classifier-free
guidance, EMA. No CLIP, no SD photo prior, no transliteration.

**Setup:** GPU T4 x2 (uses one), Internet ON.
**Optional input:** attach your Paper-1 notebook output ("Add Input" → Your Work) —
its best LoRA adapter becomes the content-fidelity judge; otherwise the base
recognizer judges.
**Time:** ~30min latent cache + ~2-2.5h per model at 150 epochs + ~1h sampling/eval
≈ **6-7h total**. Resumable: re-Run-All continues from the last checkpoint.


**Measured T4 pace: ~200s/epoch.** Trains BOTH modes in parallel (one per T4). Resume-aware: attach the checkpoint dataset and the restore cell continues from the saved epoch. Session plan: S1 = akshara 150->300 (~8.3h) & codepoint 60->~270 (hits 12h wall); S2 = codepoint ->300 + sampling + eval (~4h).\n

In [ ]:
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install "transformers==4.57.6" "diffusers>=0.31" peft datasets editdistance accelerate clean-fid
import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# ---- GPU compatibility guard (Kaggle "latest" torch drops P100/sm_60 kernels) ----
# Subprocess check so the main interpreter never imports the broken torch first.
import subprocess, sys
rc = subprocess.run([sys.executable, "-c",
    "import torch; (torch.ones(2, device='cuda')*2).sum().item(); print(torch.cuda.get_device_name(0))"],
    capture_output=True, text=True)
print(rc.stdout, rc.stderr[-500:] if rc.returncode else "")
if rc.returncode != 0:
    print("torch/GPU mismatch — installing torch 2.5.1 cu121 (supports sm_60/P100)...")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "torch==2.5.1", "torchvision==0.20.1",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
    rc2 = subprocess.run([sys.executable, "-c",
        "import torch; (torch.ones(2, device='cuda')*2).sum().item(); print('fixed:', torch.cuda.get_device_name(0))"],
        capture_output=True, text=True)
    print(rc2.stdout, rc2.stderr[-300:])
    assert rc2.returncode == 0, "GPU still broken after torch downgrade"


In [ ]:
# ---- Dataset parquets (pinned revision, resumable) ----
import os
os.makedirs("data/iiit_hindi_parquet", exist_ok=True)
BASE = "https://huggingface.co/datasets/c3rl/IIIT-INDIC-HW-WORDS-Hindi/resolve/2a27244ff5f5f5eaaf86aa4b9411beb356921f51/data"
FILES = ["train-00000-of-00003.parquet","train-00001-of-00003.parquet","train-00002-of-00003.parquet",
         "validation-00000-of-00001.parquet","test-00000-of-00001.parquet"]
for f in FILES:
    !curl -sL --fail --continue-at - --retry 10 --retry-delay 5 -o data/iiit_hindi_parquet/{f} {BASE}/{f}
!ls -la data/iiit_hindi_parquet/

In [ ]:
import os
for d in ["backend", "paper1/experiments", "paper2/ldm", "paper2/experiments/vocab"]:
    os.makedirs(d, exist_ok=True)
for f in ["backend/__init__.py", "paper1/__init__.py", "paper1/experiments/__init__.py",
          "paper2/__init__.py", "paper2/ldm/__init__.py", "paper2/experiments/__init__.py"]:
    open(f, "w").close()
print("package dirs ready")

In [ ]:
# ---- RESTORE from attached checkpoint dataset (resume support) ----
# Copies a previous session's checkpoints into place so training auto-resumes.
# Handles both layouts: extracted folders OR a single paper2_ckpts.zip.
import glob, os, shutil, subprocess

zips = sorted(glob.glob("/kaggle/input/**/paper2_ckpts.zip", recursive=True))
if zips and not glob.glob("/kaggle/input/**/ldm_akshara/checkpoint.pt", recursive=True):
    print("[restore] unzipping", zips[0], flush=True)
    os.makedirs("/kaggle/tmp_ckpts", exist_ok=True)
    subprocess.run(["unzip", "-oq", zips[0], "-d", "/kaggle/tmp_ckpts"], check=True)
    search_roots = ["/kaggle/tmp_ckpts", "/kaggle/input"]
else:
    search_roots = ["/kaggle/input"]

def restore(rel, dest):
    for root in search_roots:
        hits = sorted(glob.glob(f"{root}/**/{rel}", recursive=True))
        if hits:
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            if not os.path.exists(dest):
                print(f"[restore] {hits[0]} -> {dest}", flush=True)
                shutil.copy2(hits[0], dest)
            return

for mode in ["akshara", "codepoint"]:
    restore(f"ldm_{mode}/checkpoint.pt", f"paper2/runs/ldm_{mode}/checkpoint.pt")
    restore(f"ldm_{mode}/tokenizer.json", f"paper2/runs/ldm_{mode}/tokenizer.json")
restore("latents_train.pt", "paper2/runs/latents_train.pt")

import torch
for mode in ["akshara", "codepoint"]:
    p = f"paper2/runs/ldm_{mode}/checkpoint.pt"
    if os.path.exists(p):
        print(f"{mode}: checkpoint at epoch {torch.load(p, map_location='cpu')['epoch'] + 1}")
    else:
        print(f"{mode}: no checkpoint — trains from scratch")

In [ ]:
%%writefile backend/preprocessing.py
"""
DevGen Framework — Advanced Preprocessing Pipeline
Handles: Binarization, Deskewing, Denoising, Normalization
"""

import cv2
import numpy as np
from PIL import Image
import io


def bytes_to_cv2(image_bytes: bytes) -> np.ndarray:
    """Convert raw image bytes to OpenCV BGR image."""
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return img


def cv2_to_pil(img: np.ndarray) -> Image.Image:
    """Convert OpenCV BGR image to PIL RGB Image."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def binarize(img: np.ndarray) -> np.ndarray:
    """Adaptive binarization for handwritten documents.
    Uses Gaussian adaptive thresholding to handle uneven lighting."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, blockSize=15, C=10
    )
    return binary


def deskew(img: np.ndarray) -> np.ndarray:
    """Correct document skew using Hough Line Transform."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100,
                            minLineLength=100, maxLineGap=10)
    if lines is None:
        return img

    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if abs(angle) < 45:  # Only consider near-horizontal lines
            angles.append(angle)

    if not angles:
        return img

    median_angle = np.median(angles)
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, median_angle, 1.0)
    rotated = cv2.warpAffine(img, M, (w, h),
                              flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)
    return rotated


def denoise(img: np.ndarray) -> np.ndarray:
    """Non-local means denoising for handwritten documents."""
    if len(img.shape) == 3:
        return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    else:
        return cv2.fastNlMeansDenoising(img, None, 10, 7, 21)


def normalize_for_model(img: np.ndarray, target_height: int = 384,
                         target_width: int = 384) -> np.ndarray:
    """Resize image to target dimensions while maintaining aspect ratio.
    Pads with white if needed."""
    h, w = img.shape[:2]
    scale = min(target_height / h, target_width / w)
    new_h, new_w = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Create white canvas and center the image
    if len(img.shape) == 3:
        canvas = np.ones((target_height, target_width, 3), dtype=np.uint8) * 255
    else:
        canvas = np.ones((target_height, target_width), dtype=np.uint8) * 255

    y_offset = (target_height - new_h) // 2
    x_offset = (target_width - new_w) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
    return canvas


def crop_to_foreground(img: np.ndarray, padding_ratio: float = 0.18) -> np.ndarray:
    """Crop around visible handwriting while keeping a little context."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img

    h, w = gray.shape[:2]
    min_area = max(12, int(h * w * 0.0001))
    boxes = [cv2.boundingRect(contour) for contour in contours if cv2.contourArea(contour) >= min_area]
    if not boxes:
        return img

    x1 = min(x for x, _, _, _ in boxes)
    y1 = min(y for _, y, _, _ in boxes)
    x2 = max(x + bw for x, _, bw, _ in boxes)
    y2 = max(y + bh for _, y, _, bh in boxes)

    pad_x = max(8, int((x2 - x1) * padding_ratio))
    pad_y = max(8, int((y2 - y1) * padding_ratio))
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x)
    y2 = min(h, y2 + pad_y)

    return img[y1:y2, x1:x2]


def preprocess_cv2_for_ocr(img: np.ndarray) -> np.ndarray:
    """Word-image preprocessing on a BGR array.
    Matches HF Space Logic:
    - AR <= 1.55: Crop to foreground.
    - AR > 2.2: Crop to foreground + Pad to Square.
    - 1.55 < AR <= 2.2: No change (raw).
    """
    h, w = img.shape[:2]
    aspect_ratio = w / float(h)

    if aspect_ratio <= 1.55:
        img = crop_to_foreground(img)
    elif aspect_ratio > 2.2:
        img = crop_to_foreground(img)
        img = normalize_for_model(img, target_height=384, target_width=384)
    return img


def preprocess_for_ocr(image_bytes: bytes) -> Image.Image:
    """Prepare a word image for TrOCR from raw bytes (serving path)."""
    img = bytes_to_cv2(image_bytes)
    if img is None: return None
    return cv2_to_pil(preprocess_cv2_for_ocr(img))


def preprocess_pil_for_ocr(image: Image.Image) -> Image.Image:
    """Prepare a PIL word image for TrOCR without a bytes round-trip
    (training/eval path — must stay in parity with preprocess_for_ocr)."""
    img = cv2.cvtColor(np.array(image.convert("RGB")), cv2.COLOR_RGB2BGR)
    return cv2_to_pil(preprocess_cv2_for_ocr(img))


def full_preprocess(image_bytes: bytes) -> Image.Image:
    """Complete preprocessing pipeline for document images.
    Steps: Denoise → Deskew → Binarize → Normalize → PIL"""
    img = bytes_to_cv2(image_bytes)
    img = denoise(img)
    img = deskew(img)
    # Keep color for ViT input (TrOCR expects RGB images)
    img = normalize_for_model(img)
    return cv2_to_pil(img)


In [ ]:
%%writefile paper1/experiments/akshara.py
"""
Devanagari akshara (orthographic syllable / grapheme cluster) segmentation
and script-aware error categorization.

An akshara is the perceptual writing unit of Devanagari:
    (C halant)* C (nukta)? (matra)? (sign)*   e.g. क्ष्मी = क + ् + ष + ् + म + ी
    V (sign)*                                 independent vowel, e.g. आँ
Codepoint-level CER treats क्ष (3 codepoints) and क (1) asymmetrically;
akshara-level metrics weight them equally, matching how readers perceive errors.
"""

from __future__ import annotations

from dataclasses import dataclass

VIRAMA = "्"  # ्
NUKTA = "़"   # ़
ZWJ_ZWNJ = {"‌", "‍"}

# Unicode Devanagari block ranges
_CONSONANTS = set(
    [chr(c) for c in range(0x0915, 0x093A)]  # क..ह
    + [chr(c) for c in range(0x0958, 0x0960)]  # nukta consonants क़..य़
    + ["ॻ", "ॼ", "ॾ", "ॿ"]  # rare extensions
)
_INDEPENDENT_VOWELS = {chr(c) for c in range(0x0904, 0x0915)}  # ऄ..औ
_MATRAS = {chr(c) for c in range(0x093E, 0x094D)} | {"ॢ", "ॣ", "ऺ", "ऻ", "ॎ", "ॏ"}
_SIGNS = {"ँ", "ं", "ः"}  # candrabindu, anusvara, visarga
_DIGITS = {chr(c) for c in range(0x0966, 0x0970)}  # ०..९


def is_consonant(ch: str) -> bool:
    return ch in _CONSONANTS


def split_aksharas(text: str) -> list[str]:
    """Segment NFC-normalized Devanagari text into akshara clusters.
    Non-Devanagari characters become single-character clusters."""
    clusters: list[str] = []
    i, n = 0, len(text)
    while i < n:
        ch = text[i]
        if is_consonant(ch):
            j = i + 1
            while j < n and text[j] == NUKTA:
                j += 1
            # consume (halant [ZWJ] consonant)* chains — conjuncts
            while (
                j < n
                and text[j] == VIRAMA
                and (
                    (j + 1 < n and is_consonant(text[j + 1]))
                    or (j + 2 < n and text[j + 1] in ZWJ_ZWNJ and is_consonant(text[j + 2]))
                )
            ):
                j += 2 if is_consonant(text[j + 1]) else 3
                while j < n and text[j] == NUKTA:
                    j += 1
            # word-final dead consonant (trailing halant)
            if j < n and text[j] == VIRAMA and (j + 1 == n or not is_consonant(text[j + 1])):
                j += 1
            if j < n and text[j] in _MATRAS:
                j += 1
            while j < n and text[j] in _SIGNS:
                j += 1
            clusters.append(text[i:j])
            i = j
        elif ch in _INDEPENDENT_VOWELS:
            j = i + 1
            while j < n and text[j] in _SIGNS:
                j += 1
            clusters.append(text[i:j])
            i = j
        else:
            clusters.append(ch)
            i += 1
    return clusters


def is_conjunct(cluster: str) -> bool:
    """True if the cluster contains a consonant-joining virama (e.g. क्ष, त्र, स्थ)."""
    for k, ch in enumerate(cluster):
        if ch == VIRAMA and k + 1 < len(cluster):
            nxt = cluster[k + 1]
            if is_consonant(nxt) or nxt in ZWJ_ZWNJ:
                return True
    return False


def has_matra(cluster: str) -> bool:
    return any(ch in _MATRAS for ch in cluster)


def has_sign(cluster: str) -> bool:
    return any(ch in _SIGNS for ch in cluster)


def base_consonants(cluster: str) -> str:
    """The consonant/vowel skeleton of a cluster, stripped of matras and signs."""
    return "".join(ch for ch in cluster if is_consonant(ch) or ch in _INDEPENDENT_VOWELS or ch == VIRAMA)


@dataclass
class AksharaEdit:
    op: str  # "sub" | "ins" | "del"
    ref: str  # reference cluster ("" for insertions)
    pred: str  # predicted cluster ("" for deletions)

    def category(self) -> str:
        """Script-aware error category, checked most-specific first."""
        ref, pred = self.ref, self.pred
        if self.op == "ins":
            return "insertion_conjunct" if is_conjunct(pred) else "insertion"
        if self.op == "del":
            return "deletion_conjunct" if is_conjunct(ref) else "deletion"
        # substitution subtypes
        if is_conjunct(ref) or is_conjunct(pred):
            return "conjunct_substitution"
        if base_consonants(ref) == base_consonants(pred):
            if has_sign(ref) != has_sign(pred) and has_matra(ref) == has_matra(pred):
                return "sign_error"  # anusvara/candrabindu/visarga only
            return "matra_error"  # same skeleton, different vowel marking
        return "base_substitution"


def align_aksharas(ref_clusters: list[str], pred_clusters: list[str]) -> list[AksharaEdit]:
    """Levenshtein alignment over cluster sequences; returns only edit ops."""
    m, n = len(ref_clusters), len(pred_clusters)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref_clusters[i - 1] == pred_clusters[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)

    edits: list[AksharaEdit] = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] and ref_clusters[i - 1] == pred_clusters[j - 1]:
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            edits.append(AksharaEdit("sub", ref_clusters[i - 1], pred_clusters[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            edits.append(AksharaEdit("del", ref_clusters[i - 1], ""))
            i -= 1
        else:
            edits.append(AksharaEdit("ins", "", pred_clusters[j - 1]))
            j -= 1
    edits.reverse()
    return edits


In [ ]:
%%writefile paper1/experiments/common.py
"""
Shared utilities for Paper 1 experiments: seeding, model loading, and metrics.

All metrics operate on NFC-normalized Unicode. This matters for Devanagari:
the same visual word can be encoded with different codepoint sequences
(e.g. precomposed vs decomposed nukta forms), which silently inflates CER.
"""

from __future__ import annotations

import json
import os
import random
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import editdistance
import numpy as np
import torch
from peft import PeftModel
from transformers import AutoTokenizer, TrOCRProcessor, ViTImageProcessor, VisionEncoderDecoderModel

PROJECT_ROOT = Path(__file__).resolve().parents[2]
RESULTS_DIR = Path(__file__).resolve().parent / "results"
DATASET_NAME = "c3rl/IIIT-INDIC-HW-WORDS-Hindi"
DEFAULT_BASE_MODEL = "paudelanil/trocr-devanagari-2"
DEFAULT_IMAGE_PROCESSOR = "google/vit-base-patch16-224-in21k"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_best_torch_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def normalize_text(text: str) -> str:
    return unicodedata.normalize("NFC", text.strip())


def cer(prediction: str, reference: str) -> float:
    """Character error rate over NFC-normalized codepoints."""
    prediction = normalize_text(prediction)
    reference = normalize_text(reference)
    if not reference:
        return 0.0 if not prediction else 1.0
    return editdistance.eval(prediction, reference) / len(reference)


def word_error(prediction: str, reference: str) -> int:
    """Exact-match word error (0 = correct). Dataset is word-level, so
    aggregate word error rate == 1 - word recognition accuracy (WRA)."""
    return int(normalize_text(prediction) != normalize_text(reference))


def akshara_error_rate(prediction: str, reference: str) -> float:
    """Edit distance over akshara (grapheme cluster) sequences."""
    from paper1.experiments.akshara import split_aksharas

    pred_units = split_aksharas(normalize_text(prediction))
    ref_units = split_aksharas(normalize_text(reference))
    if not ref_units:
        return 0.0 if not pred_units else 1.0
    return editdistance.eval(pred_units, ref_units) / len(ref_units)


def bootstrap_ci(values: list[float], n_resamples: int = 1000, alpha: float = 0.05,
                 seed: int = 0) -> tuple[float, float]:
    """Percentile bootstrap CI for the mean of per-sample values."""
    rng = np.random.default_rng(seed)
    arr = np.asarray(values, dtype=np.float64)
    means = np.empty(n_resamples)
    for i in range(n_resamples):
        means[i] = rng.choice(arr, size=len(arr), replace=True).mean()
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))


@dataclass
class EvalSummary:
    run_name: str
    num_samples: int
    cer: float
    cer_ci95: tuple[float, float]
    aer: float  # akshara error rate
    word_accuracy: float
    trainable_params: Optional[int] = None
    total_params: Optional[int] = None
    config: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            "run_name": self.run_name,
            "num_samples": self.num_samples,
            "cer": round(self.cer, 5),
            "cer_ci95": [round(v, 5) for v in self.cer_ci95],
            "aer": round(self.aer, 5),
            "word_accuracy": round(self.word_accuracy, 5),
            "trainable_params": self.trainable_params,
            "total_params": self.total_params,
            "config": self.config,
        }


def load_processor(model_name: str, processor_source: Optional[str] = None) -> TrOCRProcessor:
    """Mirror backend/trocr_engine.py processor resolution so eval matches serving."""
    if processor_source:
        try:
            image_processor = ViTImageProcessor.from_pretrained(processor_source)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            return TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
        except Exception:
            pass
    try:
        return TrOCRProcessor.from_pretrained(model_name)
    except Exception as exc:
        print(f"[common] Processor fallback for '{model_name}': {exc}")
        image_processor = ViTImageProcessor.from_pretrained(DEFAULT_IMAGE_PROCESSOR)
        image_processor.image_mean = [0.5, 0.5, 0.5]
        image_processor.image_std = [0.5, 0.5, 0.5]
        image_processor.rescale_factor = 1.0 / 255.0
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        return TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)


def load_model(
    base_model: str = DEFAULT_BASE_MODEL,
    adapter_path: Optional[str] = None,
    full_model_path: Optional[str] = None,
    device: Optional[str] = None,
) -> tuple[torch.nn.Module, TrOCRProcessor, str]:
    """Load one of: base model, base+LoRA adapter, or a fully fine-tuned checkpoint.

    Returns (model, processor, device). Model is in eval mode on the device.
    """
    device = device or os.getenv("TROCR_DEVICE") or get_best_torch_device()

    if full_model_path:
        model = VisionEncoderDecoderModel.from_pretrained(full_model_path)
        processor = load_processor(full_model_path)
    else:
        processor_source = None
        if adapter_path and (Path(adapter_path) / "preprocessor_config.json").exists():
            processor_source = adapter_path
        processor = load_processor(base_model, processor_source)
        model = VisionEncoderDecoderModel.from_pretrained(base_model)
        if adapter_path:
            model = PeftModel.from_pretrained(model, adapter_path)

    base = model.get_base_model() if isinstance(model, PeftModel) else model
    base.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    base.config.pad_token_id = processor.tokenizer.pad_token_id
    base.config.eos_token_id = processor.tokenizer.sep_token_id
    base.config.vocab_size = base.config.decoder.vocab_size
    base.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
    base.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    base.generation_config.eos_token_id = processor.tokenizer.sep_token_id

    model.to(device)
    model.eval()
    return model, processor, device


def count_params(model: torch.nn.Module) -> tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    if trainable == 0 and isinstance(model, PeftModel):
        trainable = sum(
            p.numel()
            for name, p in model.named_parameters()
            if "lora_" in name or "modules_to_save" in name
        )
    return trainable, total


def save_results(payload: dict, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)
    print(f"[common] Wrote {output_path}")


In [ ]:
%%writefile paper2/ldm/tokenizer.py
"""
Content tokenizers for the Devanagari handwriting LDM.

Two modes — the paper's central ablation:
  codepoint : one token per Unicode codepoint. Conjunct formation must be
              learned implicitly from co-occurring virama sequences.
  akshara   : one token per orthographic syllable (grapheme cluster), with
              codepoint FALLBACK for clusters unseen at vocab-build time, so
              out-of-vocabulary conjuncts remain representable. Whether this
              helps or hurts unseen-conjunct generalization is the question.

Vocabulary is built from the training split. Special ids:
  0 = PAD, 1 = NULL (classifier-free guidance null condition), 2 = UNK.
"""

from __future__ import annotations

import json
from pathlib import Path

import sys
sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from paper1.experiments.akshara import split_aksharas  # noqa: E402

PAD_ID, NULL_ID, UNK_ID = 0, 1, 2
SPECIALS = ["<pad>", "<null>", "<unk>"]


class DevanagariTokenizer:
    def __init__(self, mode: str, vocab: dict[str, int], max_len: int):
        assert mode in ("codepoint", "akshara")
        self.mode = mode
        self.vocab = vocab
        self.max_len = max_len
        # codepoint entries double as the fallback space for akshara mode
        self.char_ids = {k: v for k, v in vocab.items() if len(k) == 1}

    @property
    def vocab_size(self) -> int:
        return max(self.vocab.values()) + 1

    def units(self, text: str) -> list[str]:
        return split_aksharas(text) if self.mode == "akshara" else list(text)

    def encode(self, text: str) -> list[int]:
        ids: list[int] = []
        for unit in self.units(text):
            if unit in self.vocab:
                ids.append(self.vocab[unit])
            elif self.mode == "akshara":
                # unseen cluster -> decompose to codepoints (compositional fallback)
                ids.extend(self.char_ids.get(ch, UNK_ID) for ch in unit)
            else:
                ids.append(UNK_ID)
        ids = ids[: self.max_len]
        return ids + [PAD_ID] * (self.max_len - len(ids))

    def save(self, path: str | Path) -> None:
        Path(path).write_text(
            json.dumps({"mode": self.mode, "max_len": self.max_len, "vocab": self.vocab},
                       ensure_ascii=False),
            encoding="utf-8",
        )

    @classmethod
    def load(cls, path: str | Path) -> "DevanagariTokenizer":
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        return cls(data["mode"], data["vocab"], data["max_len"])

    @classmethod
    def build(cls, texts: list[str], mode: str, min_freq: int = 2,
              max_len: int | None = None) -> "DevanagariTokenizer":
        from collections import Counter

        counts: Counter[str] = Counter()
        lengths: list[int] = []
        for text in texts:
            units = split_aksharas(text) if mode == "akshara" else list(text)
            counts.update(units)
            lengths.append(len(units))

        vocab: dict[str, int] = {tok: i for i, tok in enumerate(SPECIALS)}
        # always include every single codepoint seen (fallback space)
        chars = sorted({ch for t in texts for ch in t})
        for ch in chars:
            vocab.setdefault(ch, len(vocab))
        if mode == "akshara":
            for unit, freq in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
                if freq >= min_freq and len(unit) > 1:
                    vocab.setdefault(unit, len(vocab))

        if max_len is None:
            lengths.sort()
            max_len = min(lengths[int(0.995 * len(lengths))] + 2, 32)
        return cls(mode, vocab, max_len)


In [ ]:
%%writefile paper2/ldm/train_ldm.py
"""
Train a compact latent diffusion model FROM SCRATCH for Devanagari
handwritten word generation (Paper 2 v2). WordStylist-style recipe,
script-aware conditioning.

Design:
- Frozen SD VAE (stabilityai/sd-vae-ft-mse) maps 64x256 word images to
  4x8x32 latents. Latents are precomputed ONCE and cached (~150MB for 70k
  images), so training steps touch only the small UNet -> a full from-scratch
  run fits in a single Kaggle T4 session.
- UNet2DConditionModel (~65M params) with cross-attention over learned
  content-token embeddings (codepoint or akshara mode — the ablation).
- Classifier-free guidance: 10% of batches get the NULL condition.
- EMA weights, fp16 autocast, checkpoint/auto-resume every epoch.

Usage:
    python -m paper2.ldm.train_ldm --mode akshara --parquet-dir data/iiit_hindi_parquet \
        --output-dir paper2/runs/ldm_akshara --epochs 300
"""

from __future__ import annotations

import argparse
import io
import math
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from paper2.ldm.tokenizer import NULL_ID, PAD_ID, DevanagariTokenizer  # noqa: E402

IMG_H, IMG_W = 64, 256
LATENT_C, LATENT_H, LATENT_W = 4, IMG_H // 8, IMG_W // 8
VAE_SCALE = 0.18215
COND_DIM = 384


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def normalize_text(text: str) -> str:
    import unicodedata
    return unicodedata.normalize("NFC", str(text).strip())


def prepare_word_image(image: Image.Image) -> Image.Image:
    """Grayscale word strip -> white-padded RGB 64x256, ink-preserving."""
    img = image.convert("L")
    w, h = img.size
    scale = IMG_H / h
    new_w = max(8, min(int(w * scale), IMG_W))
    img = img.resize((new_w, IMG_H), Image.Resampling.LANCZOS)
    canvas = Image.new("L", (IMG_W, IMG_H), color=255)
    canvas.paste(img, ((IMG_W - new_w) // 2, 0))
    return canvas.convert("RGB")


class ContentEncoder(nn.Module):
    """Token embeddings + positions + a light transformer -> cross-attn memory."""

    def __init__(self, vocab_size: int, max_len: int, dim: int = COND_DIM):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, dim, padding_idx=PAD_ID)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_len, dim))
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=6, dim_feedforward=dim * 4,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        x = self.token_emb(token_ids) + self.pos_emb[:, : token_ids.shape[1]]
        pad_mask = token_ids == PAD_ID
        return self.encoder(x, src_key_padding_mask=pad_mask)


def build_unet() -> "UNet2DConditionModel":
    from diffusers import UNet2DConditionModel

    return UNet2DConditionModel(
        in_channels=LATENT_C,
        out_channels=LATENT_C,
        block_out_channels=(128, 256, 512),
        down_block_types=("CrossAttnDownBlock2D", "CrossAttnDownBlock2D", "DownBlock2D"),
        up_block_types=("UpBlock2D", "CrossAttnUpBlock2D", "CrossAttnUpBlock2D"),
        layers_per_block=2,
        cross_attention_dim=COND_DIM,
        attention_head_dim=8,
        norm_num_groups=32,
    )


def precompute_latents(parquet_dir: str, split_prefix: str, cache_path: Path,
                       tokenizer: DevanagariTokenizer, device: str,
                       batch_size: int = 64, limit: int | None = None):
    """VAE-encode the whole split once; cache latents + token ids to disk."""
    if cache_path.exists():
        blob = torch.load(cache_path, map_location="cpu")
        # token ids depend on tokenizer mode — verify match
        if blob.get("tokenizer_mode") == tokenizer.mode and blob.get("max_len") == tokenizer.max_len:
            print(f"[latents] Reusing cache {cache_path} ({blob['latents'].shape[0]} samples)")
            return blob["latents"], blob["token_ids"]
        print("[latents] Cache tokenizer mismatch — rebuilding token ids only")
        latents = blob["latents"]
        texts = blob["texts"]
        token_ids = torch.tensor([tokenizer.encode(t) for t in texts], dtype=torch.long)
        torch.save({"latents": latents, "token_ids": token_ids, "texts": texts,
                    "tokenizer_mode": tokenizer.mode, "max_len": tokenizer.max_len}, cache_path)
        return latents, token_ids

    from datasets import load_dataset
    from diffusers import AutoencoderKL

    files = sorted(str(p) for p in Path(parquet_dir).glob(f"{split_prefix}-*.parquet"))
    spec = f"train[:{limit}]" if limit else "train"
    dataset = load_dataset("parquet", data_files=files, split=spec)

    vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device).eval()
    if device == "cuda":
        vae = vae.half()

    all_latents, texts = [], []
    batch_imgs: list[torch.Tensor] = []

    def flush():
        if not batch_imgs:
            return
        x = torch.stack(batch_imgs).to(device)
        if device == "cuda":
            x = x.half()
        with torch.no_grad():
            lat = vae.encode(x).latent_dist.sample() * VAE_SCALE
        all_latents.append(lat.float().cpu())
        batch_imgs.clear()

    for i, sample in enumerate(dataset):
        img = sample["image"]
        if isinstance(img, dict) and "bytes" in img:
            img = Image.open(io.BytesIO(img["bytes"]))
        img = prepare_word_image(img)
        arr = torch.from_numpy(np.array(img, dtype=np.float32)).permute(2, 0, 1) / 127.5 - 1.0
        batch_imgs.append(arr)
        texts.append(normalize_text(sample["text"]))
        if len(batch_imgs) == batch_size:
            flush()
        if (i + 1) % 5000 == 0:
            print(f"[latents] {i + 1}/{len(dataset)}")
    flush()

    latents = torch.cat(all_latents)
    token_ids = torch.tensor([tokenizer.encode(t) for t in texts], dtype=torch.long)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"latents": latents, "token_ids": token_ids, "texts": texts,
                "tokenizer_mode": tokenizer.mode, "max_len": tokenizer.max_len}, cache_path)
    print(f"[latents] Cached {latents.shape[0]} latents -> {cache_path} "
          f"({cache_path.stat().st_size / 1e6:.0f} MB)")
    del vae
    if device == "cuda":
        torch.cuda.empty_cache()
    return latents, token_ids


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train the Devanagari handwriting LDM from scratch.")
    parser.add_argument("--mode", choices=["codepoint", "akshara"], required=True)
    parser.add_argument("--parquet-dir", default="data/iiit_hindi_parquet")
    parser.add_argument("--output-dir", required=True)
    parser.add_argument("--epochs", type=int, default=300)
    parser.add_argument("--batch-size", type=int, default=256)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--cfg-dropout", type=float, default=0.1)
    parser.add_argument("--ema-decay", type=float, default=0.9995)
    parser.add_argument("--save-every", type=int, default=10, help="Epochs between checkpoints.")
    parser.add_argument("--preview-every", type=int, default=25, help="Epochs between sample grids.")
    parser.add_argument("--train-limit", type=int, default=None)
    parser.add_argument("--latent-cache", default=None, help="Defaults to <output-dir>/../latents_train.pt")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = get_device()
    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- tokenizer (built once from the train split, cached) ----
    tok_path = out_dir / "tokenizer.json"
    if tok_path.exists():
        tokenizer = DevanagariTokenizer.load(tok_path)
    else:
        from datasets import load_dataset
        files = sorted(str(p) for p in Path(args.parquet_dir).glob("train-*.parquet"))
        spec = f"train[:{args.train_limit}]" if args.train_limit else "train"
        texts = [normalize_text(t) for t in load_dataset("parquet", data_files=files, split=spec)["text"]]
        tokenizer = DevanagariTokenizer.build(texts, mode=args.mode)
        tokenizer.save(tok_path)
    print(f"[train] mode={args.mode} vocab={tokenizer.vocab_size} max_len={tokenizer.max_len}")

    # ---- data ----
    cache = Path(args.latent_cache) if args.latent_cache else out_dir.parent / "latents_train.pt"
    latents, token_ids = precompute_latents(
        args.parquet_dir, "train", cache, tokenizer, device, limit=args.train_limit,
    )
    loader = DataLoader(
        TensorDataset(latents, token_ids),
        batch_size=args.batch_size, shuffle=True, drop_last=True,
        num_workers=2, pin_memory=device == "cuda",
    )

    # ---- model ----
    from diffusers import DDPMScheduler
    from diffusers.training_utils import EMAModel

    unet = build_unet().to(device)
    encoder = ContentEncoder(tokenizer.vocab_size, tokenizer.max_len).to(device)
    scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")
    params = list(unet.parameters()) + list(encoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=1e-4)
    ema = EMAModel(unet.parameters(), decay=args.ema_decay)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")
    n_params = sum(p.numel() for p in params)
    print(f"[train] trainable params: {n_params/1e6:.1f}M — {len(loader)} steps/epoch")

    # ---- auto-resume ----
    start_epoch = 0
    ckpt_path = out_dir / "checkpoint.pt"
    if ckpt_path.exists():
        state = torch.load(ckpt_path, map_location=device)
        unet.load_state_dict(state["unet"])
        encoder.load_state_dict(state["encoder"])
        optimizer.load_state_dict(state["optimizer"])
        ema.load_state_dict(state["ema"])
        scaler.load_state_dict(state["scaler"])
        start_epoch = state["epoch"] + 1
        # checkpoint restores the old LR; honor the CLI value for resumed epochs
        for group in optimizer.param_groups:
            group["lr"] = args.lr
        print(f"[train] Resumed from epoch {start_epoch} (lr={args.lr})")

    null_ids = torch.full((args.batch_size, tokenizer.max_len), PAD_ID, dtype=torch.long, device=device)
    null_ids[:, 0] = NULL_ID

    for epoch in range(start_epoch, args.epochs):
        unet.train()
        running = 0.0
        for step, (lat, ids) in enumerate(loader):
            lat, ids = lat.to(device, non_blocking=True), ids.to(device, non_blocking=True)
            if torch.rand(()) < args.cfg_dropout:
                ids = null_ids[: ids.shape[0]]

            noise = torch.randn_like(lat)
            timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (lat.shape[0],), device=device)
            noisy = scheduler.add_noise(lat, noise, timesteps)

            with torch.autocast(device_type=device if device != "mps" else "cpu",
                                enabled=device == "cuda"):
                cond = encoder(ids)
                pred = unet(noisy, timesteps, encoder_hidden_states=cond).sample
                loss = F.mse_loss(pred.float(), noise.float())

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            ema.step(unet.parameters())
            running += loss.item()

        avg = running / len(loader)
        print(f"[train] epoch {epoch + 1}/{args.epochs}  loss {avg:.4f}", flush=True)

        if (epoch + 1) % args.save_every == 0 or epoch + 1 == args.epochs:
            torch.save({
                "epoch": epoch, "unet": unet.state_dict(), "encoder": encoder.state_dict(),
                "optimizer": optimizer.state_dict(), "ema": ema.state_dict(),
                "scaler": scaler.state_dict(),
                "config": {"mode": args.mode, "vocab_size": tokenizer.vocab_size,
                           "max_len": tokenizer.max_len, "cond_dim": COND_DIM},
            }, ckpt_path)
            print(f"[train] checkpoint @ epoch {epoch + 1} -> {ckpt_path}")

        if (epoch + 1) % args.preview_every == 0 or epoch + 1 == args.epochs:
            try:
                from paper2.ldm.sample_ldm import generate_grid
                preview = out_dir / f"preview_epoch{epoch + 1:04d}.png"
                generate_grid(unet, encoder, ema, tokenizer, device, preview)
                print(f"[train] preview -> {preview}")
            except Exception as exc:
                print(f"[train] preview failed (non-fatal): {exc}")

    print("[train] DONE")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paper2/ldm/sample_ldm.py
"""
Sample from a trained Devanagari handwriting LDM (paper2/ldm/train_ldm.py).

Two entry points:
- generate_grid(): quick 8-word preview grid during training.
- CLI: generate one image per word from a vocab file into the extracted
  format (images/ + labels.csv) that content_fidelity.py / compute_fid.py
  and train_trocr.py --synthetic-dir consume.

Usage:
    python -m paper2.ldm.sample_ldm \
        --checkpoint paper2/runs/ldm_akshara/checkpoint.pt \
        --vocab-file paper2/experiments/vocab/oov_unseen_conjunct.txt \
        --out-dir data_synth/v2_akshara_oov_unseen --guidance 2.5
"""

from __future__ import annotations

import argparse
import csv
import sys
from pathlib import Path

import torch
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from paper2.ldm.tokenizer import NULL_ID, PAD_ID, DevanagariTokenizer  # noqa: E402

VAE_SCALE = 0.18215
LATENT_SHAPE = (4, 8, 32)  # C, H, W for 64x256 images

PREVIEW_WORDS = ["नमस्ते", "विद्यालय", "क्षत्रिय", "हिंदी", "लक्ष्मी", "भारत", "अनुसन्धान", "ज्ञान"]


@torch.no_grad()
def sample_batch(unet, encoder, tokenizer, words, device, vae=None,
                 steps: int = 50, guidance: float = 2.5, seed: int = 42):
    from diffusers import AutoencoderKL, DDIMScheduler

    if vae is None:
        vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device).eval()

    scheduler = DDIMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")
    scheduler.set_timesteps(steps, device=device)

    ids = torch.tensor([tokenizer.encode(w) for w in words], dtype=torch.long, device=device)
    null_ids = torch.full_like(ids, PAD_ID)
    null_ids[:, 0] = NULL_ID
    cond = encoder(ids)
    uncond = encoder(null_ids)

    generator = torch.Generator(device="cpu").manual_seed(seed)
    latents = torch.randn((len(words), *LATENT_SHAPE), generator=generator).to(device)
    latents = latents * scheduler.init_noise_sigma

    for t in scheduler.timesteps:
        inp = scheduler.scale_model_input(latents, t)
        eps_c = unet(inp, t, encoder_hidden_states=cond).sample
        eps_u = unet(inp, t, encoder_hidden_states=uncond).sample
        eps = eps_u + guidance * (eps_c - eps_u)
        latents = scheduler.step(eps, t, latents).prev_sample

    images = vae.decode(latents / VAE_SCALE).sample
    images = ((images.clamp(-1, 1) + 1) * 127.5).permute(0, 2, 3, 1).cpu().numpy().astype("uint8")
    return [Image.fromarray(arr) for arr in images], vae


def generate_grid(unet, encoder, ema, tokenizer, device, out_path: Path,
                  words=PREVIEW_WORDS, steps: int = 30) -> None:
    """Training-time preview using EMA weights (restored afterwards)."""
    unet.eval()
    ema.store(unet.parameters())
    ema.copy_to(unet.parameters())
    try:
        # guidance 1.5 — the 2.5 default oversaturates and made previews unreadable
        images, _ = sample_batch(unet, encoder, tokenizer, words, device, steps=steps, guidance=1.5)
        rows = len(images)
        grid = Image.new("RGB", (256, 64 * rows), "white")
        for i, img in enumerate(images):
            grid.paste(img, (0, i * 64))
        grid.save(out_path)
    finally:
        ema.restore(unet.parameters())
        unet.train()


def load_model(checkpoint: str, device: str, use_ema: bool = True):
    from diffusers.training_utils import EMAModel
    from paper2.ldm.train_ldm import ContentEncoder, build_unet

    state = torch.load(checkpoint, map_location=device)
    cfg = state["config"]
    tokenizer = DevanagariTokenizer.load(Path(checkpoint).parent / "tokenizer.json")

    unet = build_unet().to(device)
    unet.load_state_dict(state["unet"])
    encoder = ContentEncoder(cfg["vocab_size"], cfg["max_len"]).to(device)
    encoder.load_state_dict(state["encoder"])
    if use_ema:
        ema = EMAModel(unet.parameters())
        ema.load_state_dict(state["ema"])
        ema.copy_to(unet.parameters())
    unet.eval()
    encoder.eval()
    return unet, encoder, tokenizer, state["epoch"] + 1


def main() -> None:
    parser = argparse.ArgumentParser(description="Generate word images from a trained LDM.")
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--vocab-file", required=True)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--steps", type=int, default=50)
    parser.add_argument("--guidance", type=float, default=2.5)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--repeat", type=int, default=1, help="Images per word (distinct seeds).")
    parser.add_argument("--no-ema", action="store_true")
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    unet, encoder, tokenizer, epoch = load_model(args.checkpoint, device, use_ema=not args.no_ema)
    print(f"[sample] {tokenizer.mode} model @ epoch {epoch} on {device}")

    words = [w.strip() for w in Path(args.vocab_file).read_text(encoding="utf-8").splitlines() if w.strip()]
    tasks = [(w, r) for r in range(args.repeat) for w in words]

    out_dir = Path(args.out_dir)
    images_dir = out_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    rows: list[dict[str, str]] = []
    vae = None
    for start in range(0, len(tasks), args.batch_size):
        chunk = tasks[start : start + args.batch_size]
        seed = args.seed + start
        images, vae = sample_batch(
            unet, encoder, tokenizer, [w for w, _ in chunk], device, vae=vae,
            steps=args.steps, guidance=args.guidance, seed=seed,
        )
        for (word, rep), img in zip(chunk, images):
            filename = f"gen_{len(rows):06d}.png"
            img.save(images_dir / filename)
            rows.append({"filename": filename, "text": word, "seed": str(seed), "repeat": str(rep)})
        print(f"[sample] {min(start + args.batch_size, len(tasks))}/{len(tasks)}")

    with (out_dir / "labels.csv").open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=["filename", "text", "seed", "repeat"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"[sample] Wrote {len(rows)} images -> {out_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paper2/experiments/content_fidelity.py
"""
Content fidelity for Paper 2: read every generated image with a Devanagari
TrOCR recognizer (best Paper-1 adapter) and score CER/AER/word accuracy
against the intended text. High scores mean generations are readable as the
right word.

Input: a generated set in extracted format (images/ + labels.csv from
paper1/experiments/generate_synthetic.py).

Usage:
    python -m paper2.experiments.content_fidelity \
        --generated-dir data_synth/eval_iv \
        --adapter-path paper1/runs/lora_r16_attn \
        --run-name fidelity_iv
"""

from __future__ import annotations

import argparse
import csv
import sys
from pathlib import Path

import torch
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from paper1.experiments.common import (  # noqa: E402
    DEFAULT_BASE_MODEL,
    EvalSummary,
    akshara_error_rate,
    bootstrap_ci,
    cer,
    load_model,
    normalize_text,
    save_results,
    set_seed,
    word_error,
)

RESULTS_DIR = Path(__file__).resolve().parent / "results"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Content fidelity of generated handwriting.")
    parser.add_argument("--generated-dir", required=True, help="Directory with images/ + labels.csv.")
    parser.add_argument("--base-model", default=DEFAULT_BASE_MODEL)
    parser.add_argument("--adapter-path", default=None, help="Judge recognizer LoRA adapter.")
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--num-beams", type=int, default=4)
    parser.add_argument("--max-length", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--run-name", required=True)
    parser.add_argument("--app-preprocess", action="store_true", default=True,
                        help="Apply the recognizer's crop/pad preprocessing (default on).")
    parser.add_argument("--output-dir", default=str(RESULTS_DIR))
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    set_seed(args.seed)

    model, processor, device = load_model(
        base_model=args.base_model, adapter_path=args.adapter_path,
    )

    generated_dir = Path(args.generated_dir)
    with (generated_dir / "labels.csv").open("r", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    print(f"[content_fidelity] {len(rows)} generated images from {generated_dir}")

    preprocess = None
    if args.app_preprocess:
        from backend.preprocessing import preprocess_pil_for_ocr as preprocess

    samples: list[dict] = []
    for start in range(0, len(rows), args.batch_size):
        batch = rows[start : start + args.batch_size]
        images = [Image.open(generated_dir / "images" / r["filename"]).convert("RGB") for r in batch]
        if preprocess:
            images = [preprocess(img) for img in images]
        references = [str(r["text"]) for r in batch]

        pixel_values = processor(images=images, return_tensors="pt").pixel_values.to(device)
        with torch.inference_mode():
            output_ids = model.generate(
                inputs=pixel_values,
                max_length=args.max_length,
                num_beams=args.num_beams,
                early_stopping=True,
            )
        predictions = processor.batch_decode(output_ids, skip_special_tokens=True)

        for row, pred, ref in zip(batch, predictions, references):
            samples.append({
                "filename": row["filename"],
                "reference": normalize_text(ref),
                "prediction": normalize_text(pred),
                "cer": round(cer(pred, ref), 5),
                "aer": round(akshara_error_rate(pred, ref), 5),
                "word_error": word_error(pred, ref),
            })

    cers = [s["cer"] for s in samples]
    summary = EvalSummary(
        run_name=args.run_name,
        num_samples=len(samples),
        cer=sum(cers) / len(cers),
        cer_ci95=bootstrap_ci(cers, seed=args.seed),
        aer=sum(s["aer"] for s in samples) / len(samples),
        word_accuracy=1.0 - sum(s["word_error"] for s in samples) / len(samples),
        config={
            "generated_dir": str(generated_dir),
            "judge_base_model": args.base_model,
            "judge_adapter": args.adapter_path,
            "num_beams": args.num_beams,
            "seed": args.seed,
        },
    )
    print(f"[content_fidelity] {args.run_name}: CER {summary.cer:.4f}  "
          f"AER {summary.aer:.4f}  WAcc {summary.word_accuracy:.4f}")
    save_results({"summary": summary.to_dict(), "samples": samples},
                 Path(args.output_dir) / f"{args.run_name}.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paper2/experiments/compute_fid.py
"""
FID between generated word images and real test-split word images.

Uses clean-fid (pip install clean-fid). Real images are exported once from
the parquet test split to a folder; FID is then folder-vs-folder.

Usage:
    python -m paper2.experiments.compute_fid \
        --generated-dir data_synth/eval_iv/images \
        --parquet-dir data/iiit_hindi_parquet --real-limit 5000
"""

from __future__ import annotations

import argparse
import io
import json
import sys
from pathlib import Path

from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

RESULTS_DIR = Path(__file__).resolve().parent / "results"
REAL_CACHE = Path(__file__).resolve().parent / "real_test_images"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="FID for generated handwriting.")
    parser.add_argument("--generated-dir", required=True, help="Folder of generated PNGs.")
    parser.add_argument("--parquet-dir", default="data/iiit_hindi_parquet")
    parser.add_argument("--real-limit", type=int, default=5000)
    parser.add_argument("--size", type=int, default=256, help="Resize both sides to this square.")
    parser.add_argument("--run-name", default=None)
    return parser.parse_args()


def export_real_images(parquet_dir: str, limit: int, size: int) -> Path:
    out_dir = REAL_CACHE / f"n{limit}_s{size}"
    if out_dir.exists() and len(list(out_dir.glob("*.png"))) >= limit:
        print(f"[fid] Reusing cached real images at {out_dir}")
        return out_dir
    from datasets import load_dataset

    files = sorted(str(p) for p in Path(parquet_dir).glob("test-*.parquet"))
    dataset = load_dataset("parquet", data_files=files, split=f"train[:{limit}]")
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, sample in enumerate(dataset):
        img = sample["image"]
        if isinstance(img, dict) and "bytes" in img:
            img = Image.open(io.BytesIO(img["bytes"]))
        img.convert("RGB").resize((size, size), Image.Resampling.LANCZOS).save(
            out_dir / f"real_{i:06d}.png"
        )
    print(f"[fid] Exported {limit} real images to {out_dir}")
    return out_dir


def main() -> None:
    args = parse_args()
    from cleanfid import fid as cleanfid

    real_dir = export_real_images(args.parquet_dir, args.real_limit, args.size)

    # Resize generated copies to the same square for a fair comparison
    generated_dir = Path(args.generated_dir)
    resized_dir = generated_dir.parent / f"{generated_dir.name}_fid{args.size}"
    resized_dir.mkdir(exist_ok=True)
    pngs = sorted(generated_dir.glob("*.png"))
    for path in pngs:
        target = resized_dir / path.name
        if not target.exists():
            Image.open(path).convert("RGB").resize(
                (args.size, args.size), Image.Resampling.LANCZOS
            ).save(target)

    score = cleanfid.compute_fid(str(real_dir), str(resized_dir))
    run_name = args.run_name or generated_dir.parent.name
    print(f"[fid] {run_name}: FID = {score:.2f}  "
          f"({len(pngs)} generated vs {args.real_limit} real @ {args.size}px)")

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    out = RESULTS_DIR / f"fid_{run_name}.json"
    out.write_text(json.dumps({
        "run_name": run_name,
        "fid": round(score, 4),
        "n_generated": len(pngs),
        "n_real": args.real_limit,
        "size": args.size,
    }, indent=2), encoding="utf-8")
    print(f"[fid] Wrote {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paper2/experiments/vocab/iv.txt
कॉलोनियों
अष्टम
नेगी
दूरगामी
तिकरित
गुज़ारे
किषोर
सूखने
करते।
मैडम
आजमाइए
आएगा
कहॉं
तय
दयनीय
संतों
अस्सी
हटता।
झुकना
सुसरालजनों
मेघा
तस्दीक
रेल्वे
न्योता
अजूबा
चिकित्सक
मैम
बच
न्यूज
चरवाहे
ढांचागत
फैशन
किले
कहा-ज़रा
भाग्यसाली
काम-धाम
बिंदू
बढ़ेगी
नाराजगी
आहाता
लाद
सिमट
खींचनेवाले
भर्तृहरि
कंकड़-पत्थर
सौंदर्यीकरण
पवित्रता
बीएसई
११
जुड़ी
एक्सेप्ट
इंस्टी
त्रस्त
पराजय
कंप्यूटरों
दयावंत
किकबॉक्सिंग
भाजी
न्यूट्रालिटी
लगने
बुलावे
चीरते-फाड़ते
बैठेगा
बागान
डाकघर
निगाहे-रहगुज़र
एनजीटी
छूटनी
सिंदरी
दूध
चुनाव
लिफ़्ट
भाईयों
निर्गुन
स्पेक
तलब
प्रासंगिकता
उठाना
थानवी
आज़ाद।
प्रणाम
महॢष
निदान
ऊपरी
डिट
हानिकारक
प्रगाढ़ता
डू
शेरवानी
मर्म
लागने
गैर-चुनावी
नार्वे
गुजारने
देखूँगी
हटा
सीरा
नाच
युगल
महानिदेशक
बीजू
तर्क
गिल्ली-डंडे
सकता
शवों
कलासाधकों
इजाज़त
कैदियों
चकमेड़री
चिट्ठियों
मैथ्यू
उभरे
भेजे
भाष्य
लोप
साख
द्विवेदी
स्कैन
अनिन्द्य
क्यूं
सिलावट
निगमों
बगीचों
कोई-कोई
पष्चिम
रपट
चालकों
लगान
अकबर
नानासाहेब
शौक
जरूरत
संवत्सर
कुष्ठ
पाटलिपुत्र
झुका
ब्राडकास्टर
चिल्लाते
सीसोलाभाजपा
साधारण
अंगरेजी
प्रार्थना
वॉयस
अम्ल
कोटवार
बुकलेट
पूज्यनीय
दिनेश
उत्तर-पूर्व
दिशाहीनता
हाय
औषध
कमाल
वीर्य
एक्टरों
सार्वजनिक
खुलेआम
खोला
वार्षिक
सोंचा
चेल्म्सफोर्ड
नाले
साँस
मॉर्टगेज
डिस्क
सीसा
टाइपिंग
पैसों
महात्म्य
ब्रदर्स
राऊ
समझाना
रौब
ख़ानदान
देवपूजा
तुड़वा
उम्रकैद
फ्रैंकलिन
अर्थोपार्जन
स्टील
थू।
तल्प
अण्डमान-निकोबार
एतद्द्वारा
उत्सव
थर्राया।
एंटरप्राईजेस
आग्रहों
फ़ेंकना।
एडिटिंग
सनातन
दिखने
पंक्तियाँ
वीएलएसआई
ड्यूटियां
सीरीयल
गम्भीरता
हुइबे
होर्स
वही


In [ ]:
%%writefile paper2/experiments/vocab/oov_seen.txt
अंज़ोहैर्याहे
ऋड़ाणाकं
एज्ञायुक्राष्ठा
ऐंफॉटों
कंत्तीब्बा
कंधोंटॉ
क़द्यारूठोंडॉ
क़ायोंल्लो
क़ीसीब्रर्ताष
कांटे
कीग्रील्मी
कुंनींलींफ़ु
क्टितों
क्ता५ने
क्रल्यूच्चादौ
क्रोस्तेहौ
क्लिनंखु
क्सर्देक्टिलौ
खिभौ
खींभूप्ताहैं
खेंझू
खोदौ
ख्यात्रवों
गांद्धांस्टेग्स
गांप्रीऋता
गाझफेंसोर्चा
गीफ़ा
गुंरोर्दे
गेआंग्रीर्किम्मे
ग्याल्यूपासौच्चा
ग्सडेखी
घारकैंफ्टगुं
घिक्ष्मीऌ
चिंस्कोर्तीथी
चीदीकट्रै
चैप्रीम्पर्नि
चोजिट्टेशाप्प
च्चास्याठीर्बा
छाल्दीरोंमेतै
छुग्रा
छोनं
जिघिॠआँ
जीर्जा
जेंल्कयेविद्दे
जेन्द्रष्ठतेंप्ता
जैब्रावेझ
ज्यर्व
टेहीद
ट्टान्दुग्नि
ट्टीआं
ट्रांक़ा
ट्सबा
ठागाँप्सहिपौ
डर्नशि
ड़ोनो७संफ़ि
ड़ोम्हवेंमाँकै
डेढ़िलंशा
ड्डीरअँऑहै
ढ़न्दाहींर्ती०
ढ़िडॉ
ढ़ोस्लियींष्य
ढाङवांर्म
ढेक्ताटें८
णीआंप्सतौचै
ण्डठीख़
ण्यभची
ताल्डढूंभ्यहाँ
तैखूत्याल्मी
तोचदे
त्तह्मल्मी
त्मप्पटे
त्मिस्तिराहूं
त्रीटेंत्रोंॠ
त्वथिनौज़ोईं
त्वर्द्रपिह्ला
त्वश्च
त्सशूप्लेगिस्प
थफ़ेटॉछांगीं
थियेलोभ
थीढूंष्ठान्द्रटों
थीनींगौझांढ
दाजैस्कृर्वे
दुएझू
दृदइंस्वी
देबुऊफंवीं
दैधूदौहूँत्प्रे
द्रत्प्रेएंद्धिगं
द्राशुठेल्दीटु
धूगोस्लि
धोंत्राछांमेर्थी
धोन्टफॉप्री
नत्याजोन्हा
नूबाक्षढे
नेंदुर्द्रनि
नैर्यत्रों
नॉक्क
नोंन्टफ्तब्रा
नौड्सत्प्रेब्याजी
न्कपृ
न्कह्ला
न्तितुडेशांक्लि
न्द्रपिं
न्द्रस्मखी
न्नर्ममंडोप्या
न्नातबों
न्यूज़ीक्टिच्ची
पिमैधेन्द
पोब्रात्थतः
प्रिगैर्तितें
प्रोन्यू
प्लेश
फंत्तेन्तुर्फड़
फ़ेख़तौ५वीं
फाद्दे
फ्तत्रिमॉर्मीट
बंज्ज्वऊ
बंस्मृसुंर्चिऔ
बिंबों
बोड़ी
भीक़पभ्य
भोभेर्चम्पत्ते
भोष्यशंतैब्बा
भौहचों
मांचालं
मैद्यपॉ
मैब्रांन्याख़ास्वी
म्मेबॉम्यसं
म्हजुझाऋखी
म्हाखों।१
म३
यंत्यढाजं
यीग्रीउंप्ति
योडे
रूंठोन्नज्यो
रॉफीश्य
रौप्री
र्करक्ष्मीड्डीड़ि
र्जेवृ
र्टल्टीस्टो
र्टेजिंओब्रि
र्डखींट्रै
र्णरांन्यारेत्र
र्तिचोणी
र्दिऌ
र्दिर्टज्ञा
र्दीर्वाझीभित्मि
र्निस्कृभैचिगाँ
र्पओंभिलीश्च
र्याखी
र्याजूप्रेघाग़
र्यार्त
र्शर्न
र्शल्मप्रूढ़ोसी
लास्त्यरीवोचें
लिंर्द्रते२थ
लूंग्री६
लैहाष्णयं०
लोंयी
ल्दीप्रिम्बड़ीबे
ल्मीर्ती
ल्लाऽध्दडोंच्च
ल्सड़ोक
विच्च
विनींर्वा
वेघिज्ञ
वेरैर्र
वैर्तावैएँ
वैल्लफेंफि
व्याम्मीदंफ
शांटेय
शेगृ
शैस्टोस्वडिछे
श्चाइ
श्रक्ष्णषिनिंति
श्रिल्लाठमे
श्रीत्रात्राष्ट्र
श्रीम्पर्किद्देपै
ष्ट्रपि
ष्ट्रीफ़ी
ष्णहृणोंतंदु
सिंष्कात्त
सीस्लिस्ते
सृलॉम्रयें
सोंब्बा
स्टड़ोंर्क
स्तेक्षेस्टोपं
स्तेड़ोंझ
स्त्रीढ़ार्ती
स्थन्दुरौफ्तर्रा
स्थाबूठफ्यूओ
स्यारांगाड़ेबि
स्वीलौट्रो
स्सास्कर्थी
हँम्मी
हंर्मबूऋ९
हाँथास्ट्रोचंधा
हालांन्दुहूंफ़ु
।क्राटडी
२टिमूं
७दैड़


In [ ]:
%%writefile paper2/experiments/vocab/oov_unseen_conjunct.txt
अद्चीखि
इङ्हबै
ईह्खेग
ईह्निगे
एंठ्ञेदु
एंढ्छोल
एऱ्घेच
एस्जम
ओंट्बासी
ओंठ्लुगो
कघ्डूजे
कांऩ्वीशी
काथ्खेरो
कीण्लाह
कृक्चिक
केध्ङखि
केऩ्कोया
केऱ्गोति
कैट्षेसु
कोळ्सीदि
खब्ऩोरि
खाफ्ङम
खाळ्नूअ
खुन्पीरि
खेप्गका
ख्जोदू
गंऩ्दिउ
गीफ्योलो
गेश्गाट
चान्षीनि
चीफ्शूई
चुय्ऩोकृ
चेञ्ठोख
चेन्गूध
च्बुता
च्लेजी
छ्डेफ़
जञ्शागा
ज़प्ठीलों
ज़ाय्शिओ
जेढ्खिले
ज्खीति
झझ्जीथ
झम्नुकु
झाऩ्गेये
झाप्ङूटि
झाम्घावे
झाऴ्नोनं
झ्थोभू
झ्पीसा
ञ्ङिच
ञ्थिनो
टध्भोड
ट्तीठा
ट्शेनि
ठ्ढथा
ठ्युड
डीण्पाकै
डीण्ऴूरा
डीव्ऩुई
ड्ङेसं
ड्फिसे
ढ्केबू
ढ्चूखु
ढ्झपै
ढ्नोओ
णष्सेन
ण्चूमू
तीघ्ळाफि
तेण्तमि
तोंण्षेझ
तोंद्ळोबु
त्णुपू
त्बाको
थळ्ञिन
थान्छाचु
थाप्खनी
दीक्चेलि
दुप्गिट
दुब्तूसे
दुर्बूयु
दुव्खीद
देज्दबै
देद्ञालि
दोत्ञही
द्भुड़ा
द्भोडे
धव्ङोबू
धाछ्गेन
धाज्नष
नाध्ऱापो
नेऱ्फिह
नोंऩ्मुजी
नोंफ्ञनि
नोंय्खूटा
न्लूरं
ऩ्खिगा
पंभ्पोचा
पंस्ङिनि
पाळ्बेडि
पिघ्थुशी
पुछ्णूड़
पेञ्मीगो
पैऩ्होई
फव्लिलि
फ़ठ्छूग
फ्दीले
बग्धेबी
बव्डोगी
बुञ्ढीते
बैव्छोरो
बोद्लाची
ब्धूहो
ब्पूबं
भग्फिपा
भट्झूटी
भण्खामो
भऱ्घूए
भष्ङाव
भिण्रूमो
भूठ्ऴुव
भ्वीजि
माछ्रेमू
माव्बेवि
मीख्भूकृ
मीश्भझ
मुव्टोपा
मूब्शोफि
मेंट्टुच
मेंढ्ङान
मेळ्धिलो
यल्साशी
याफ्छूकु
याल्षोडी
येझ्टशि
येण्गूथ
योंप्नुपै
य्वाड़ा
रंञ्पोमि
रंन्णोयाँ
रद्ऱेलों
रस्शूव
राढ्ताबै
राल्शझ
रिध्नीदी
रुट्दाने
रूत्धाचु
रूह्जुपै
रेंद्ञोरें
रेंऱ्घुवे
रेञ्नेग
रेञ्ळिगी
रेभ्ऱाजा
रोंण्ङोरें
रोश्गूमू
र्घूटा
र्ठोघ
र्मोबे
लीठ्फोया
लीऱ्पुपि
लीह्खिरो
लोख्कोबा
लोछ्ऩोनी
ळ्ङाटी
ळ्छानों
ळ्लूसो
ळ्लोगं
ळ्ऴमू
ऴ्टेते
ऴ्रुखे
वञ्ङेघ
वप्ळुभू
वर्पाज़
वात्गिदू
वाम्ऴोकृ
विङ्दड़ा
वीझ्तूला
वीण्षोनी
वीन्णुबै
वैट्ऩुबू
वैढ्थागु
वैर्ळुवे
व्धूमो
शल्टूला
शिञ्शोसु
श्ञाकां
ष्घानं
ष्ङूपू
ष्झोडी
संज्नूढ़
सच्दाकों
सिंळ्तेया
सिड्ठोधा
सेड्ऱीअ
सोभ्बूक
स्पोटी
हाट्फमै
हाब्वूशि
हेछ्गद
ह्ञीठा


In [ ]:
# ---- Sanity: tokenizer ablation behaves as designed ----
import sys; sys.path.insert(0, ".")
from paper2.ldm.tokenizer import DevanagariTokenizer, PAD_ID, UNK_ID
texts = ["नमस्ते", "क्षत्रिय", "विद्यालय", "हिंदी", "लक्ष्मी", "भारत"] * 3
tok = DevanagariTokenizer.build(texts, mode="akshara", min_freq=2)
seen = [i for i in tok.encode("क्ष") if i != PAD_ID]
unseen = [i for i in tok.encode("क्भा") if i != PAD_ID]
assert len(seen) == 1 and UNK_ID not in unseen and len(unseen) == 4
print("OK: seen conjunct = 1 akshara token; unseen conjunct decomposes to codepoints")

In [ ]:
# ==================== CONFIG ====================
MODES = ["akshara", "codepoint"]   # the ablation; drop one to halve runtime
EPOCHS = 800                        # ~200s/epoch; resume-aware; legibility emerges late (flat MSE is normal)
SEED = 42
PARQUET = "data/iiit_hindi_parquet"
GUIDANCE = 1.5   # local sweep on epoch-150 ckpt: 2.5+ oversaturates, 1.0-1.5 best
STRATA = ["iv", "oov_seen", "oov_unseen_conjunct"]

# Optional judge adapter from the Paper-1 notebook output (attach as input)
import glob, os
JUDGE = None
for pat in ["/kaggle/input/*/paper1/runs/lora_r16_attn/adapter_config.json",
            "/kaggle/input/*/paper1_runs/lora_r16_attn/adapter_config.json",
            "/kaggle/input/**/lora_r16_attn/adapter_config.json"]:
    hits = glob.glob(pat, recursive=True)
    if hits:
        JUDGE = os.path.dirname(hits[0]); break
print("judge adapter:", JUDGE or "none (base recognizer will judge)")

In [ ]:
# ==================== TRAIN BOTH MODELS (parallel — one per GPU) ====================
import subprocess, time, shutil, os, torch

n_gpu = torch.cuda.device_count()
print("GPUs:", n_gpu)

# Per-mode latent cache: two processes must NOT share one cache file
# (mode mismatch triggers a rewrite -> corruption race).
for mode in MODES:
    dst = f"paper2/runs/latents_train_{mode}.pt"
    if not os.path.exists(dst) and os.path.exists("paper2/runs/latents_train.pt"):
        shutil.copy2("paper2/runs/latents_train.pt", dst)
        print("cache copy ->", dst)

procs = []
for i, mode in enumerate(MODES):
    env = dict(os.environ)
    if n_gpu > 1:
        env["CUDA_VISIBLE_DEVICES"] = str(i % n_gpu)
    log = open(f"train_{mode}.log", "w")
    cmd = (f"python -m paper2.ldm.train_ldm --mode {mode} "
           f"--parquet-dir {PARQUET} --output-dir paper2/runs/ldm_{mode} "
           f"--epochs {EPOCHS} --seed {SEED} --batch-size 256 --lr 1e-4 "
           f"--latent-cache paper2/runs/latents_train_{mode}.pt "
           f"--save-every 10 --preview-every 25")
    print(">>", f"[gpu{i % max(n_gpu,1)}]", cmd, flush=True)
    procs.append([mode, subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, env=env), log, time.time()])

done = set()
while len(done) < len(procs):
    time.sleep(300)
    for rec in procs:
        mode, p, log, t0 = rec
        if mode in done:
            continue
        lines = [l for l in open(f"train_{mode}.log").read().splitlines() if l.strip()]
        tail = lines[-1] if lines else "..."
        print(f"[{(time.time()-t0)/60:.0f}min] {mode}: {tail}", flush=True)
        if p.poll() is not None:
            done.add(mode)
            log.close()
            print(f"=== ldm_{mode}: {'OK' if p.returncode == 0 else 'FAILED'} in {(time.time()-t0)/60:.0f} min ===", flush=True)
            if p.returncode != 0:
                print(open(f"train_{mode}.log").read()[-3000:], flush=True)
                raise RuntimeError(f"training failed for {mode} — stopping notebook")


In [ ]:
# ==================== SAMPLE EVALUATION SETS ====================
import subprocess
def sh(cmd):
    print(">>", cmd, flush=True)
    return subprocess.call(cmd, shell=True)
import os
for mode in MODES:
    ckpt = f"paper2/runs/ldm_{mode}/checkpoint.pt"
    if not os.path.exists(ckpt):
        print(f"skip {mode}: no checkpoint"); continue
    for stratum in STRATA:
        repeat = 5 if stratum == "iv" else 1   # iv x5 -> 1000 imgs for stable FID
        out = f"data_synth/v2_{mode}_{stratum}"
        if os.path.exists(f"{out}/labels.csv"):
            print(f"skip {out}"); continue
        sh(f"python -m paper2.ldm.sample_ldm --checkpoint {ckpt} "
           f"--vocab-file paper2/experiments/vocab/{stratum}.txt "
           f"--out-dir {out} --guidance {GUIDANCE} --steps 50 --repeat {repeat} --seed {SEED}")

In [ ]:
# ==================== EVALUATE: content fidelity + FID ====================
for mode in MODES:
    for stratum in STRATA:
        gen = f"data_synth/v2_{mode}_{stratum}"
        if not os.path.exists(f"{gen}/labels.csv"): continue
        judge_arg = f"--adapter-path {JUDGE}" if JUDGE else ""
        sh(f"python -m paper2.experiments.content_fidelity --generated-dir {gen} "
           f"{judge_arg} --run-name v2_{mode}_{stratum} --batch-size 32")
    if os.path.exists(f"data_synth/v2_{mode}_iv/images"):
        sh(f"python -m paper2.experiments.compute_fid "
           f"--generated-dir data_synth/v2_{mode}_iv/images "
           f"--run-name v2_{mode}_iv --parquet-dir {PARQUET} --real-limit 5000")

In [ ]:
# ==================== RESULTS SUMMARY ====================
import json, glob
print(f"{'run':30} {'CER':>7} {'AER':>7} {'WAcc':>7}")
for f in sorted(glob.glob("paper2/experiments/results/v2_*.json")):
    s = json.load(open(f))["summary"]
    print(f"{s['run_name']:30} {s['cer']:>7.4f} {s['aer']:>7.4f} {s['word_accuracy']:>7.4f}")
print()
for f in sorted(glob.glob("paper2/experiments/results/fid_*.json")):
    d = json.load(open(f))
    print(f"FID {d['run_name']}: {d['fid']}")
print()
print("Reading guide: iv = in-vocab quality; oov_seen = novel word forms;")
print("oov_unseen_conjunct = THE ablation — akshara vs codepoint on unseen ligatures.")

In [ ]:
# ==================== PACKAGE FOR DOWNLOAD ====================
!rm -f /kaggle/working/paper2_artifacts.zip /kaggle/working/paper2_models.zip
!zip -qr /kaggle/working/paper2_artifacts.zip \
    paper2/experiments/results \
    paper2/runs/*/tokenizer.json \
    paper2/runs/*/preview_*.png \
    data_synth/v2_*_oov_unseen_conjunct 2>/dev/null
!zip -qr /kaggle/working/paper2_models.zip paper2/runs/*/checkpoint.pt 2>/dev/null
!ls -la /kaggle/working/*.zip
print("paper2_artifacts.zip = results + previews + unseen-conjunct samples (small)")
print("paper2_models.zip = full checkpoints (~1.6GB) — download to continue training later")